In [ ]:
# ============================================================================
# ENTERPRISE MORTGAGE DOCUMENT INTELLIGENCE SYSTEM
# Handles 200+ page PDFs with production-grade reliability
# ============================================================================

# ============================================================================
# INSTALLATION
# ============================================================================
print("Installing dependencies...")
# Install required Python packages
!pip install -q gradio groq pymupdf pdfplumber pypdf2 sentence-transformers faiss-cpu pytesseract pillow pdf2image easyocr opencv-python-headless python-dotenv
# Install Tesseract OCR and Poppler utilities
!apt-get install -q tesseract-ocr poppler-utils

print("✓ Installation complete")

Installing dependencies...
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.11).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.
✓ Installation complete


In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================
# Import Gradio for the user interface
import gradio as gr
# Import Groq for the language model
from groq import Groq
# Import PyMuPDF for fast PDF processing
import fitz  # PyMuPDF
# Import pdfplumber for text and table extraction
import pdfplumber
# Import PyPDF2 as a standard PDF fallback
import PyPDF2
# Import pytesseract for OCR
import pytesseract
# Import easyocr for advanced OCR
import easyocr
# Import pdf2image to convert PDF pages to images for OCR
from pdf2image import convert_from_path
# Import OpenCV for image processing
import cv2
# Import standard libraries
import os
import time
import traceback
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
from datetime import datetime
import numpy as np
# Import SentenceTransformer for embeddings
from sentence_transformers import SentenceTransformer
# Import FAISS for vector indexing
import faiss
import re
from collections import defaultdict
import json
import logging
from pathlib import Path

In [ ]:
# ============================================================================
# LOGGING CONFIGURATION
# ============================================================================
# Configure basic logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
# Get a logger instance
logger = logging.getLogger(__name__)

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    """System configuration"""

    # API Keys
    GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"

    # Model configurations
    EMBEDDING_MODELS = [
        "sentence-transformers/all-mpnet-base-v2",  # Best quality
        "sentence-transformers/all-MiniLM-L12-v2",  # Balanced
        "sentence-transformers/all-MiniLM-L6-v2"    # Fast fallback
    ]

    GROQ_MODELS = [
        "llama-3.1-70b-versatile",  # Primary Groq model
        "mixtral-8x7b-32768",       # Fallback Groq model
        "llama-3.1-8b-instant"      # Fast fallback Groq model
    ]

    # Processing parameters
    CHUNK_SIZE = 600 # Size of text chunks
    CHUNK_OVERLAP = 100 # Overlap between chunks
    MAX_PAGES_PER_BATCH = 50 # Maximum pages to process in a batch (not used in current process_document)
    OCR_CONFIDENCE_THRESHOLD = 0.6 # Minimum confidence for OCR results
    SIMILARITY_THRESHOLD = 0.3 # Minimum similarity score for retrieval

    # Performance (not fully implemented in current version)
    MAX_WORKERS = 4
    TIMEOUT_PER_PAGE = 30

In [ ]:
# ============================================================================
# DATA STRUCTURES
# ============================================================================

@dataclass
class ProcessingStatus:
    """Real-time processing status"""
    stage: str # Current processing stage
    progress: float # Progress as a float between 0 and 1
    message: str # Status message
    current_page: int # Current page being processed
    total_pages: int # Total pages in the document
    elapsed_time: float # Time elapsed since start
    estimated_remaining: float # Estimated time remaining
    errors: List[str] = field(default_factory=list) # List of errors encountered
    warnings: List[str] = field(default_factory=list) # List of warnings

@dataclass
class PageData:
    """Data for a single page"""
    page_num: int # Page number (0-indexed)
    text: str # Extracted text
    extraction_method: str # Method used for extraction
    confidence: float # Confidence score of extraction
    processing_time: float # Time taken to process the page
    has_tables: bool # Whether the page contains tables
    has_images: bool # Whether the page contains images (not currently detected)
    word_count: int # Number of words in the extracted text
    error: Optional[str] = None # Error message if extraction failed

@dataclass
class DocumentChunk:
    """Enhanced chunk with detailed metadata"""
    text: str # Text content of the chunk
    doc_id: str # Unique document ID
    doc_name: str # Original document filename
    chunk_id: int # Unique chunk ID within the document
    page_start: int # Starting page number (1-indexed)
    page_end: int # Ending page number (1-indexed)
    char_start: int # Starting character index within the original page text
    char_end: int # Ending character index within the original page text
    extraction_method: str # Method used for extraction of the source page
    confidence: float # Confidence of extraction of the source page
    has_tables: bool # Whether the source page had tables
    embedding: Optional[np.ndarray] = None # Vector embedding of the chunk
    metadata: Dict[str, Any] = field(default_factory=dict) # Additional metadata

@dataclass
class DocumentMetadata:
    """Complete document metadata"""
    doc_id: str # Unique document ID
    filename: str # Original filename
    total_pages: int # Total pages in the document
    total_chunks: int # Total chunks created
    processing_time: float # Total time to process the document
    extraction_methods: Dict[str, int] # Count of pages processed by each method
    avg_confidence: float # Average extraction confidence across pages
    file_size_mb: float # File size in megabytes
    has_errors: bool # Whether errors occurred during processing
    error_pages: List[int] # List of page numbers with errors
    timestamp: str # Processing timestamp

In [ ]:
# ============================================================================
# ADVANCED PDF PROCESSOR
# ============================================================================

class AdvancedPDFProcessor:
    """
    Multi-tier PDF processing with intelligent fallback
    Handles complex layouts, scanned documents, and error recovery
    """

    def __init__(self):
        # Initialize EasyOCR reader lazily
        self.easyocr_reader = None
        # Track methods attempted and succeeded
        self.methods_attempted = defaultdict(int)
        self.methods_succeeded = defaultdict(int)

    def _init_easyocr(self):
        """Lazy initialization of EasyOCR"""
        if self.easyocr_reader is None:
            try:
                # Initialize EasyOCR with English language
                self.easyocr_reader = easyocr.Reader(['en'])
                logger.info("EasyOCR initialized successfully")
            except Exception as e:
                logger.warning(f"EasyOCR initialization failed: {e}")
                self.easyocr_reader = False

    def extract_text_pymupdf(self, pdf_path: str, page_num: int) -> Tuple[str, float]:
        """Extract text using PyMuPDF (fastest, most reliable)"""
        try:
            doc = fitz.open(pdf_path)
            page = doc[page_num]
            text = page.get_text()
            doc.close()

            # Return text and high confidence if text is substantial
            if len(text.strip()) > 50:
                return text, 0.95
            return "", 0.0

        except Exception as e:
            logger.debug(f"PyMuPDF failed on page {page_num}: {e}")
            return "", 0.0

    def extract_text_pdfplumber(self, pdf_path: str, page_num: int) -> Tuple[str, float, bool]:
        """Extract text using pdfplumber (better for tables)"""
        try:
            with pdfplumber.open(pdf_path) as pdf:
                page = pdf.pages[page_num]
                text = page.extract_text()
                tables = page.extract_tables()

                has_tables = len(tables) > 0

                # Append table data to text
                if tables:
                    table_text = "\n\n[TABLE DATA]\n"
                    for table in tables:
                        for row in table:
                            table_text += " | ".join([str(cell) if cell else "" for cell in row]) + "\n"
                    text = text + table_text if text else table_text

                # Return text, confidence, and table status if text is substantial
                if len(text.strip()) > 50:
                    return text, 0.90, has_tables
                return "", 0.0, has_tables

        except Exception as e:
            logger.debug(f"pdfplumber failed on page {page_num}: {e}")
            return "", 0.0, False

    def extract_text_pypdf2(self, pdf_path: str, page_num: int) -> Tuple[str, float]:
        """Extract text using PyPDF2 (standard fallback)"""
        try:
            with open(pdf_path, 'rb') as file:
                pdf = PyPDF2.PdfReader(file)
                page = pdf.pages[page_num]
                text = page.extract_text()

                # Return text and confidence if text is substantial
                if len(text.strip()) > 50:
                    return text, 0.85
                return "", 0.0

        except Exception as e:
            logger.debug(f"PyPDF2 failed on page {page_num}: {e}")
            return "", 0.0

    def extract_text_tesseract(self, pdf_path: str, page_num: int) -> Tuple[str, float]:
        """Extract text using Tesseract OCR"""
        try:
            # Convert PDF page to image
            images = convert_from_path(pdf_path, first_page=page_num+1, last_page=page_num+1)
            if not images:
                return "", 0.0

            # Preprocess image for OCR
            img = np.array(images[0])
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            # Adaptive thresholding
            processed = cv2.adaptiveThreshold(
                gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
            )

            # Perform OCR using Tesseract
            text = pytesseract.image_to_string(processed)

            # Get confidence from Tesseract data
            data = pytesseract.image_to_data(processed, output_type=pytesseract.Output.DICT)
            confidences = [int(conf) for conf in data['conf'] if conf != '-1']
            avg_confidence = np.mean(confidences) / 100 if confidences else 0.0

            # Return text and confidence if text is substantial and confidence is above threshold
            if len(text.strip()) > 50 and avg_confidence > Config.OCR_CONFIDENCE_THRESHOLD:
                return text, avg_confidence
            return "", 0.0

        except Exception as e:
            logger.debug(f"Tesseract OCR failed on page {page_num}: {e}")
            return "", 0.0

    def extract_text_easyocr(self, pdf_path: str, page_num: int) -> Tuple[str, float]:
        """Extract text using EasyOCR (best for complex layouts)"""
        try:
            self._init_easyocr()

            if self.easyocr_reader is False:
                return "", 0.0

            # Convert PDF page to image
            images = convert_from_path(pdf_path, first_page=page_num+1, last_page=page_num+1)
            if not images:
                return "", 0.0

            img = np.array(images[0])

            # Preprocess image
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            denoised = cv2.fastNlMeansDenoising(gray)

            # Perform OCR using EasyOCR
            results = self.easyocr_reader.readtext(denoised)

            text_parts = []
            confidences = []

            # Collect text and confidences from EasyOCR results
            for (bbox, text, conf) in results:
                text_parts.append(text)
                confidences.append(conf)

            text = " ".join(text_parts)
            avg_confidence = np.mean(confidences) if confidences else 0.0

            # Return text and confidence if text is substantial and confidence is above threshold
            if len(text.strip()) > 50 and avg_confidence > Config.OCR_CONFIDENCE_THRESHOLD:
                return text, avg_confidence
            return "", 0.0

        except Exception as e:
            logger.debug(f"EasyOCR failed on page {page_num}: {e}")
            return "", 0.0

    def process_single_page(self, pdf_path: str, page_num: int) -> PageData:
        """
        Process a single page with intelligent fallback cascade
        Attempts multiple extraction methods in order of reliability
        """
        start_time = time.time()
        # Define extraction methods and their order
        methods = [
            ("PyMuPDF", self.extract_text_pymupdf),
            ("pdfplumber", self.extract_text_pdfplumber),
            ("PyPDF2", self.extract_text_pypdf2),
            ("Tesseract", self.extract_text_tesseract),
            ("EasyOCR", self.extract_text_easyocr)
        ]

        best_result = None
        best_confidence = 0.0
        best_method = "None"
        has_tables = False
        errors = []

        # Iterate through methods and try to extract text
        for method_name, method_func in methods:
            try:
                self.methods_attempted[method_name] += 1

                if method_name == "pdfplumber":
                    text, confidence, has_tables = method_func(pdf_path, page_num)
                else:
                    text, confidence = method_func(pdf_path, page_num)

                # Update best result if current method is better
                if confidence > best_confidence:
                    best_result = text
                    best_confidence = confidence
                    best_method = method_name

                    # Stop if a high confidence result is found
                    if confidence > 0.9:
                        self.methods_succeeded[method_name] += 1
                        break

            except Exception as e:
                error_msg = f"{method_name} error: {str(e)}"
                errors.append(error_msg)
                logger.debug(error_msg)

        processing_time = time.time() - start_time

        if best_result:
            self.methods_succeeded[best_method] += 1

        # Return PageData object with results
        return PageData(
            page_num=page_num,
            text=best_result or "",
            extraction_method=best_method,
            confidence=best_confidence,
            processing_time=processing_time,
            has_tables=has_tables,
            has_images=False,  # Could be enhanced
            word_count=len(best_result.split()) if best_result else 0,
            error="; ".join(errors) if not best_result else None
        )

    def process_document(self, pdf_path: str, progress_callback=None) -> Tuple[List[PageData], DocumentMetadata]:
        """
        Process entire document with real-time progress tracking
        """
        start_time = time.time()
        file_size_mb = os.path.getsize(pdf_path) / (1024 * 1024)

        # Get total pages
        try:
            doc = fitz.open(pdf_path)
            total_pages = len(doc)
            doc.close()
        except:
            raise ValueError(f"Cannot open PDF file: {pdf_path}")

        logger.info(f"Processing {total_pages} pages from {Path(pdf_path).name}")

        pages_data = []
        error_pages = []
        extraction_methods = defaultdict(int)

        # Process each page
        for page_num in range(total_pages):
            page_start_time = time.time()

            # Process single page
            page_data = self.process_single_page(pdf_path, page_num)
            pages_data.append(page_data)

            # Track metrics
            extraction_methods[page_data.extraction_method] += 1
            if page_data.error:
                error_pages.append(page_num)

            # Call progress callback if provided
            if progress_callback:
                elapsed = time.time() - start_time
                avg_time_per_page = elapsed / (page_num + 1)
                remaining_pages = total_pages - (page_num + 1)
                estimated_remaining = avg_time_per_page * remaining_pages

                status = ProcessingStatus(
                    stage="extracting_text",
                    progress=(page_num + 1) / total_pages,
                    message=f"Processing page {page_num + 1}/{total_pages} using {page_data.extraction_method}",
                    current_page=page_num + 1,
                    total_pages=total_pages,
                    elapsed_time=elapsed,
                    estimated_remaining=estimated_remaining,
                    errors=[page_data.error] if page_data.error else [],
                    warnings=[]
                )
                progress_callback(status)

        # Calculate and return metadata
        total_time = time.time() - start_time
        confidences = [p.confidence for p in pages_data if p.confidence > 0]
        avg_confidence = np.mean(confidences) if confidences else 0.0

        metadata = DocumentMetadata(
            doc_id=f"doc_{int(time.time())}",
            filename=Path(pdf_path).name,
            total_pages=total_pages,
            total_chunks=0,  # Will be updated after chunking
            processing_time=total_time,
            extraction_methods=dict(extraction_methods),
            avg_confidence=float(avg_confidence),
            file_size_mb=file_size_mb,
            has_errors=len(error_pages) > 0,
            error_pages=error_pages,
            timestamp=datetime.now().isoformat()
        )

        logger.info(f"Document processed: {total_pages} pages in {total_time:.2f}s")
        logger.info(f"Extraction methods used: {dict(extraction_methods)}")
        logger.info(f"Average confidence: {avg_confidence:.2%}")

        if error_pages:
            logger.warning(f"Failed pages: {error_pages}")

        return pages_data, metadata

In [ ]:
# ============================================================================
# ADVANCED TEXT CHUNKER
# ============================================================================

class SemanticChunker:
    """
    Semantic-aware chunking that preserves context and structure
    """

    @staticmethod
    def chunk_text(text: str, chunk_size: int, overlap: int, page_num: int) -> List[Tuple[str, int, int]]:
        """
        Create semantically meaningful chunks with overlap
        Returns: List of (chunk_text, char_start, char_end)
        """
        # Clean and split text into sentences
        text = re.sub(r'\s+', ' ', text).strip()

        if not text:
            return []

        sentences = re.split(r'([.!?]\s+)', text)

        # Reconstruct sentences including delimiters
        full_sentences = []
        for i in range(0, len(sentences) - 1, 2):
            full_sentences.append(sentences[i] + (sentences[i+1] if i+1 < len(sentences) else ''))

        if len(sentences) % 2 == 1:
            full_sentences.append(sentences[-1])

        chunks = []
        current_chunk = ""
        current_start = 0
        char_position = 0

        # Create chunks based on sentences and size limits
        for sentence in full_sentences:
            sentence_len = len(sentence)

            # If adding sentence exceeds chunk size, finalize current chunk and start new one
            if len(current_chunk) + sentence_len > chunk_size and current_chunk:
                chunks.append((
                    current_chunk.strip(),
                    current_start,
                    current_start + len(current_chunk)
                ))

                # Add overlap to the new chunk
                overlap_text = current_chunk[-overlap:] if len(current_chunk) > overlap else current_chunk
                current_chunk = overlap_text + sentence
                current_start = char_position - len(overlap_text)
            else:
                if not current_chunk:
                    current_start = char_position
                current_chunk += sentence

            char_position += sentence_len

        # Add the last chunk
        if current_chunk.strip():
            chunks.append((
                current_chunk.strip(),
                current_start,
                char_position
            ))

        return chunks

    @staticmethod
    def create_chunks_from_pages(pages_data: List[PageData], doc_id: str, doc_name: str) -> List[DocumentChunk]:
        """Create DocumentChunk objects from processed PageData"""
        all_chunks = []
        chunk_counter = 0

        # Iterate through pages and create chunks
        for page_data in pages_data:
            if not page_data.text or len(page_data.text.strip()) < 50:
                continue

            page_chunks = SemanticChunker.chunk_text(
                page_data.text,
                Config.CHUNK_SIZE,
                Config.CHUNK_OVERLAP,
                page_data.page_num
            )

            # Create DocumentChunk objects for each page chunk
            for chunk_text, char_start, char_end in page_chunks:
                chunk = DocumentChunk(
                    text=chunk_text,
                    doc_id=doc_id,
                    doc_name=doc_name,
                    chunk_id=chunk_counter,
                    page_start=page_data.page_num + 1,
                    page_end=page_data.page_num + 1,
                    char_start=char_start,
                    char_end=char_end,
                    extraction_method=page_data.extraction_method,
                    confidence=page_data.confidence,
                    has_tables=page_data.has_tables,
                    metadata={
                        'word_count': len(chunk_text.split()),
                        'processing_time': page_data.processing_time
                    }
                )
                all_chunks.append(chunk)
                chunk_counter += 1

        return all_chunks

In [ ]:
# ============================================================================
# ADVANCED EMBEDDING ENGINE
# ============================================================================

class EmbeddingEngine:
    """
    Multi-model embedding engine with automatic fallback
    """

    def __init__(self):
        self.model = None
        self.model_name = None
        # Load the best available model on initialization
        self._load_best_model()

    def _load_best_model(self):
        """Load the best available embedding model from config"""
        for model_name in Config.EMBEDDING_MODELS:
            try:
                logger.info(f"Attempting to load: {model_name}")
                # Load Sentence Transformer model
                self.model = SentenceTransformer(model_name)
                self.model_name = model_name
                logger.info(f"✓ Loaded embedding model: {model_name}")
                return
            except Exception as e:
                logger.warning(f"Failed to load {model_name}: {e}")

        # Raise error if no model could be loaded
        raise RuntimeError("Could not load any embedding model")

    def encode_batch(self, texts: List[str], batch_size: int = 32, progress_callback=None) -> np.ndarray:
        """
        Encode texts in batches with progress tracking
        """
        total_texts = len(texts)
        all_embeddings = []

        # Encode texts in batches
        for i in range(0, total_texts, batch_size):
            batch = texts[i:i + batch_size]
            batch_embeddings = self.model.encode(batch, show_progress_bar=False)
            all_embeddings.append(batch_embeddings)

            # Call progress callback if provided
            if progress_callback:
                progress = (i + len(batch)) / total_texts
                status = ProcessingStatus(
                    stage="generating_embeddings",
                    progress=progress,
                    message=f"Generating embeddings: {i + len(batch)}/{total_texts}",
                    current_page=i + len(batch),
                    total_pages=total_texts,
                    elapsed_time=0,
                    estimated_remaining=0
                )
                progress_callback(status)

        # Stack embeddings into a single numpy array
        return np.vstack(all_embeddings)

    def get_dimension(self) -> int:
        """Get embedding dimension"""
        return self.model.get_sentence_embedding_dimension()

In [ ]:
# ============================================================================
# RAG SYSTEM
# ============================================================================

class ProductionRAGSystem:
    """
    Production-grade RAG system with comprehensive error handling
    """

    def __init__(self, groq_api_key: str):
        # Initialize PDF processor and Embedding engine
        self.pdf_processor = AdvancedPDFProcessor()
        self.embedding_engine = EmbeddingEngine()
        # Initialize Groq client if API key is provided
        self.groq_client = Groq(api_key=groq_api_key) if groq_api_key else None

        # Lists to store document chunks and metadata
        self.chunks: List[DocumentChunk] = []
        self.index: Optional[faiss.IndexFlatL2] = None # FAISS index for similarity search
        self.documents: List[DocumentMetadata] = []

        logger.info("RAG System initialized")
        logger.info(f"Embedding model: {self.embedding_engine.model_name}")
        logger.info(f"Embedding dimension: {self.embedding_engine.get_dimension()}")

    def process_document(self, pdf_path: str, progress_callback=None) -> Dict:
        """
        Process a document with comprehensive progress tracking
        Includes text extraction, chunking, embedding, and indexing
        """
        try:
            # Stage 1: Text Extraction
            if progress_callback:
                progress_callback(ProcessingStatus(
                    stage="starting",
                    progress=0.0,
                    message="Initializing document processing...",
                    current_page=0,
                    total_pages=0,
                    elapsed_time=0.0,
                    estimated_remaining=0.0
                ))

            start_time = time.time()
            filename = Path(pdf_path).name

            # Extract text from PDF using the PDF processor
            pages_data, doc_metadata = self.pdf_processor.process_document(
                pdf_path,
                progress_callback
            )

            # Stage 2: Chunking
            if progress_callback:
                progress_callback(ProcessingStatus(
                    stage="chunking",
                    progress=0.5,
                    message="Creating semantic chunks...",
                    current_page=0,
                    total_pages=len(pages_data),
                    elapsed_time=time.time() - start_time,
                    estimated_remaining=0.0
                ))

            # Create chunks from extracted page data
            chunks = SemanticChunker.create_chunks_from_pages(
                pages_data,
                doc_metadata.doc_id,
                filename
            )

            doc_metadata.total_chunks = len(chunks)

            # Stage 3: Embedding Generation
            if chunks:
                chunk_texts = [c.text for c in chunks]
                # Generate embeddings for chunks
                embeddings = self.embedding_engine.encode_batch(
                    chunk_texts,
                    progress_callback=progress_callback
                )

                # Assign embeddings to chunks
                for chunk, embedding in zip(chunks, embeddings):
                    chunk.embedding = embedding

            # Add processed chunks and document metadata to the system
            self.chunks.extend(chunks)
            self.documents.append(doc_metadata)

            # Stage 4: Index Building
            if progress_callback:
                progress_callback(ProcessingStatus(
                    stage="indexing",
                    progress=0.9,
                    message="Building vector index...",
                    current_page=0,
                    total_pages=0,
                    elapsed_time=time.time() - start_time,
                    estimated_remaining=0.0
                ))

            # Rebuild the FAISS index
            self._rebuild_index()

            total_time = time.time() - start_time

            # Final status update
            if progress_callback:
                progress_callback(ProcessingStatus(
                    stage="complete",
                    progress=1.0,
                    message=f"Processing complete! {len(chunks)} chunks indexed.",
                    current_page=doc_metadata.total_pages,
                    total_pages=doc_metadata.total_pages,
                    elapsed_time=total_time,
                    estimated_remaining=0.0,
                    warnings=[f"Failed pages: {doc_metadata.error_pages}"] if doc_metadata.error_pages else []
                ))

            # Return processing summary
            return {
                'success': True,
                'doc_id': doc_metadata.doc_id,
                'filename': filename,
                'pages': doc_metadata.total_pages,
                'chunks': len(chunks),
                'processing_time': total_time,
                'extraction_methods': doc_metadata.extraction_methods,
                'avg_confidence': doc_metadata.avg_confidence,
                'error_pages': doc_metadata.error_pages
            }

        except Exception as e:
            # Handle and log errors
            error_msg = f"Document processing failed: {str(e)}\n{traceback.format_exc()}"
            logger.error(error_msg)

            # Update status with error
            if progress_callback:
                progress_callback(ProcessingStatus(
                    stage="error",
                    progress=0.0,
                    message="Processing failed",
                    current_page=0,
                    total_pages=0,
                    elapsed_time=0.0,
                    estimated_remaining=0.0,
                    errors=[error_msg]
                ))

            # Return error details
            return {
                'success': False,
                'error': error_msg
            }

    def _rebuild_index(self):
        """Rebuild FAISS index from current chunks"""
        if not self.chunks:
            return

        # Get embeddings from chunks
        embeddings = np.array([c.embedding for c in self.chunks])
        dimension = embeddings.shape[1]

        # Create and add embeddings to FAISS index
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(embeddings)

        logger.info(f"Index built: {len(self.chunks)} chunks")

    def retrieve(self, query: str, k: int = 5) -> List[Tuple[DocumentChunk, float]]:
        """Retrieve relevant chunks based on query similarity"""
        if not self.chunks or self.index is None:
            return []

        # Encode the query
        query_embedding = self.embedding_engine.model.encode([query])[0]

        # Search the FAISS index
        distances, indices = self.index.search(
            np.array([query_embedding]),
            min(k * 2, len(self.chunks)) # Search more to filter later
        )

        results = []
        # Collect results above similarity threshold
        for dist, idx in zip(distances[0], indices[0]):
            if idx < len(self.chunks):
                chunk = self.chunks[idx]
                similarity = 1 / (1 + dist) # Convert distance to similarity

                if similarity > Config.SIMILARITY_THRESHOLD:
                    results.append((chunk, similarity))

                    # Stop once k relevant chunks are found
                    if len(results) >= k:
                        break

        return results

    def generate_answer(self, query: str, k: int = 5) -> Dict:
        """Generate answer using RAG (Retrieval Augmented Generation)"""
        if not self.groq_client:
            return {
                'answer': "Error: Groq API key not configured",
                'sources': [],
                'confidence': 0.0
            }

        start_time = time.time()

        # Retrieve relevant chunks
        retrieved = self.retrieve(query, k=k)

        if not retrieved:
            return {
                'answer': "No relevant information found in the uploaded documents.",
                'sources': [],
                'confidence': 0.0,
                'retrieval_time': time.time() - start_time
            }

        # Build context for the LLM from retrieved chunks
        context_parts = []
        sources = []

        for i, (chunk, score) in enumerate(retrieved, 1):
            context_parts.append(
                f"[Source {i} - Page {chunk.page_start}, {chunk.doc_name}, "
                f"Confidence: {chunk.confidence:.0%}, Method: {chunk.extraction_method}]\n{chunk.text}\n"
            )

            sources.append({
                'doc_name': chunk.doc_name,
                'page': chunk.page_start,
                'score': float(score),
                'confidence': float(chunk.confidence),
                'method': chunk.extraction_method,
                'has_tables': chunk.has_tables
            })

        context = "\n---\n".join(context_parts)

        # Build prompt for the LLM
        prompt = f"""You are a mortgage document analysis assistant. Answer the question based strictly on the provided document excerpts.

Context from documents:
{context}

Question: {query}

Instructions:
- Answer using ONLY information from the context above
-- Answer using ONLY information from the context above
        - Cite specific sources (e.g., "According to Source 1 from page 5...")
        - If information is insufficient, state this clearly
        - For mortgage-specific questions, be precise with numbers, dates, and terms
        - If you see low confidence or OCR warnings, mention uncertainty

Answer:"""

        # Generate answer using Groq LLM with fallback models
        answer = None
        generation_error = None
        model_used = None

        for model in Config.GROQ_MODELS:
            try:
                response = self.groq_client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.2,
                    max_tokens=1024
                )
                answer = response.choices[0].message.content.strip()
                model_used = model
                logger.info(f"Answer generated using {model}")
                break # Stop if generation is successful

            except Exception as e:
                logger.warning(f"Model {model} failed: {e}")
                generation_error = str(e)
                continue # Try next model

        if not answer:
            return {
                'answer': f"Failed to generate answer. Error: {generation_error}",
                'sources': sources,
                'confidence': 0.0,
                'retrieval_time': time.time() - start_time
            }

        # Calculate overall confidence based on retrieval scores and extraction confidences
        avg_confidence = np.mean([s['score'] * s['confidence'] for s in sources])

        # Return the generated answer and sources
        return {
            'answer': answer,
            'sources': sources,
            'confidence': float(avg_confidence),
            'retrieval_time': time.time() - start_time,
            'model_used': model_used,
            'chunks_retrieved': len(retrieved)
        }

    def get_system_stats(self) -> Dict:
        """Get comprehensive system statistics"""
        if not self.documents:
            return {'total_documents': 0}

        # Calculate various statistics
        total_pages = sum(d.total_pages for d in self.documents)
        total_processing_time = sum(d.processing_time for d in self.documents)

        extraction_methods = defaultdict(int)
        for doc in self.documents:
            for method, count in doc.extraction_methods.items():
                extraction_methods[method] += count

        confidences = [d.avg_confidence for d in self.documents]

        # Return statistics dictionary
        return {
            'total_documents': len(self.documents),
            'total_pages': total_pages,
            'total_chunks': len(self.chunks),
            'total_processing_time': total_processing_time,
            'avg_pages_per_doc': total_pages / len(self.documents) if len(self.documents) > 0 else 0,
            'avg_processing_time': total_processing_time / len(self.documents) if len(self.documents) > 0 else 0,
            'extraction_methods': dict(extraction_methods),
            'avg_confidence': float(np.mean(confidences)) if confidences else 0.0,
            'embedding_model': self.embedding_engine.model_name,
            'embedding_dimension': self.embedding_engine.get_dimension(),
            'index_size': self.index.ntotal if self.index else 0
        }

NameError: name 'Dict' is not defined

In [ ]:
# ============================================================================
# INITIALIZATION
# ============================================================================

# Set your Groq API key here
GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"  # Get free key at: https://console.groq.com/

if not GROQ_API_KEY:
    print("⚠️ WARNING: Groq API key not set!")
    print("Get your free API key at: https://console.groq.com/")
    print("Set it in the GROQ_API_KEY variable above")

# Initialize the RAG system
rag_system = ProductionRAGSystem(GROQ_API_KEY)

print("=" * 80)
print("MORTGAGE DOCUMENT INTELLIGENCE SYSTEM - READY")
print("=" * 80)
print(f"Embedding Model: {rag_system.embedding_engine.model_name}")
print(f"Embedding Dimension: {rag_system.embedding_engine.get_dimension()}")
print(f"Groq API: {'Configured' if GROQ_API_KEY else 'NOT CONFIGURED'}")
print("=" * 80)

MORTGAGE DOCUMENT INTELLIGENCE SYSTEM - READY
Embedding Model: sentence-transformers/all-mpnet-base-v2
Embedding Dimension: 768
Groq API: Configured


In [ ]:
# ============================================================================
# GRADIO INTERFACE FUNCTIONS
# ============================================================================

def format_status_message(status: ProcessingStatus) -> str:
    """Format processing status for display in the UI"""
    if status.stage == "starting":
        return status.message

    # Create a simple progress bar string
    progress_bar = "█" * int(status.progress * 40) + "░" * (40 - int(status.progress * 40))

    # Build the status message string
    msg = f"**Stage:** {status.stage.replace('_', ' ').title()}\n\n"
    msg += f"**Progress:** [{progress_bar}] {status.progress:.1%}\n\n"
    msg += f"**Status:** {status.message}\n\n"

    if status.total_pages > 0:
        msg += f"**Pages:** {status.current_page}/{status.total_pages}\n\n"

    msg += f"**Elapsed Time:** {status.elapsed_time:.1f}s\n"

    if status.estimated_remaining > 0:
        msg += f"**Estimated Remaining:** {status.estimated_remaining:.1f}s\n"

    # Add errors and warnings if present
    if status.errors:
        msg += f"\n**Errors:**\n"
        for error in status.errors[-3:]:  # Show last 3 errors
            msg += f"- {error}\n"

    if status.warnings:
        msg += f"\n**Warnings:**\n"
        for warning in status.warnings[-3:]:
            msg += f"- {warning}\n"

    return msg

def process_pdf_ui(files, progress=gr.Progress()):
    """Process uploaded PDFs with real-time progress updates for the UI"""
    if not files:
        return "⚠️ No files uploaded"

    results = []

    # Process each uploaded file
    for file in files:
        filename = Path(file.name).name
        progress(0, desc=f"Starting {filename}...")

        status_messages = []

        # Define the callback function to update UI status
        def progress_callback(status: ProcessingStatus):
            msg = format_status_message(status)
            status_messages.append(msg)
            progress(status.progress, desc=status.message)

        # Call the main document processing function
        result = rag_system.process_document(file.name, progress_callback)

        # Format and append results for each file
        if result['success']:
            summary = f"**✓ {filename}**\n\n"
            summary += f"- **Pages:** {result['pages']}\n"
            summary += f"- **Chunks:** {result['chunks']}\n"
            summary += f"- **Processing Time:** {result['processing_time']:.2f}s\n"
            summary += f"- **Avg Time/Page:** {result['processing_time']/result['pages']:.2f}s" if result['pages'] > 0 else f"- **Avg Time/Page:** N/A" # Avoid division by zero
            summary += f"- **Confidence:** {result['avg_confidence']:.1%}\n"
            summary += f"- **Methods Used:** {', '.join(f'{k}({v})' for k, v in result['extraction_methods'].items())}\n"

            if result['error_pages']:
                summary += f"- **⚠️ Failed Pages:** {result['error_pages']}\n"

            results.append(summary)
        else:
            results.append(f"**✗ {filename}**\n\nError: {result['error']}")

    # Get and append system statistics
    stats = rag_system.get_system_stats()

    stats_summary = f"\n\n**📊 SYSTEM STATISTICS**\n\n"
    stats_summary += f"- Total Documents: {stats['total_documents']}\n"
    stats_summary += f"- Total Pages: {stats['total_pages']}\n"
    stats_summary += f"- Total Chunks: {stats['total_chunks']}\n"
    stats_summary += f"- Avg Processing Time: {stats['avg_processing_time']:.2f}s/doc\n"
    stats_summary += f"- System Confidence: {stats['avg_confidence']:.1%}\n"
    stats_summary += f"- Embedding Model: {stats['embedding_model']}\n"

    # Return combined results and stats
    return "\n\n---\n\n".join(results) + stats_summary

def chat_ui(message, history, k_value):
    """Handle chat interactions and generate responses"""
    if not message.strip():
        return history

    # Check if documents are processed
    if not rag_system.chunks:
        history.append((message, "⚠️ Please upload and process documents first."))
        return history

    # Generate answer using the RAG system
    result = rag_system.generate_answer(message, k=k_value)

    # Format the response for the chatbot
    response = result['answer']

    # Add sources if available
    if result['sources']:
        response += f"\n\n**📚 Sources ({len(result['sources'])}):**\n"
        for i, source in enumerate(result['sources'], 1):
            response += (
                f"\n{i}. **{source['doc_name']}** (Page {source['page']})\n"
                f"   - Relevance: {source['score']:.1%}\n"
                f"   - Extraction: {source['method']} (Confidence: {source['confidence']:.1%})\n"
            )
            if source['has_tables']:
                response += f"   - Contains tables\n"

    # Add performance metrics
    response += f"\n\n**⚡ Performance:**\n"
    response += f"- Query Time: {result['retrieval_time']:.2f}s\n"
    response += f"- Model: {result.get('model_used', 'N/A')}\n"
    response += f"- Chunks Retrieved: {result['chunks_retrieved']}\n"
    response += f"- Overall Confidence: {result['confidence']:.1%}\n"

    # Append message and response to chat history
    history.append((message, response))
    return history

def get_stats_ui():
    """Get and format system statistics for UI display"""
    stats = rag_system.get_system_stats()

    if stats['total_documents'] == 0:
        return "No documents processed yet."

    # Build the statistics report string
    report = "# SYSTEM STATISTICS\n\n"
    report += f"**Documents:** {stats['total_documents']}\n"
    report += f"**Total Pages:** {stats['total_pages']}\n"
    report += f"**Total Chunks:** {stats['total_chunks']}\n"
    report += f"**Avg Pages/Doc:** {stats['avg_pages_per_doc']:.1f}\n"
    report += f"**Total Processing Time:** {stats['total_processing_time']:.2f}s\n"
    report += f"**Avg Processing Time:** {stats['avg_processing_time']:.2f}s/doc\n"
    report += f"**System Confidence:** {stats['avg_confidence']:.1%}\n\n"

    report += f"**Extraction Methods:**\n"
    for method, count in stats['extraction_methods'].items():
        report += f"- {method}: {count} pages\n"

    report += f"\n**Embedding Configuration:**\n"
    report += f"- Model: {stats['embedding_model']}\n"
    report += f"- Dimension: {stats['embedding_dimension']}\n"
    report += f"- Index Size: {stats['index_size']} vectors\n"

    # Add PDF processor specific stats
    report += f"\n**PDF Processing Stats:**\n"
    for method in ['PyMuPDF', 'pdfplumber', 'PyPDF2', 'Tesseract', 'EasyOCR']:
        attempted = rag_system.pdf_processor.methods_attempted[method]
        succeeded = rag_system.pdf_processor.methods_succeeded[method]
        if attempted > 0:
            success_rate = (succeeded / attempted) * 100
            report += f"- {method}: {succeeded}/{attempted} ({success_rate:.1f}% success)\n"

    return report

def clear_system():
    """Reset the RAG system and clear UI elements"""
    global rag_system
    # Re-initialize the RAG system
    rag_system = ProductionRAGSystem(GROQ_API_KEY)
    # Return empty chat history and a reset message
    return [], "System reset. All documents cleared."

In [ ]:
# ============================================================================
# BUILD INTERFACE
# ============================================================================

# Define the Gradio Blocks interface
with gr.Blocks(
    title="Mortgage Document Intelligence System",
    theme=gr.themes.Soft(primary_hue="slate", secondary_hue="blue"),
) as demo:

    # Markdown for title and description
    gr.Markdown("""
    # 📄 Mortgage Document Intelligence System

    **Enterprise-grade document processing for 200+ page PDFs**

    Advanced features: Multi-tier extraction • Real-time progress • Semantic chunking • Source attribution
    """)

    # Arrange UI elements in rows and columns
    with gr.Row():
        # LEFT: Document Management Column
        with gr.Column(scale=1):
            gr.Markdown("### 📤 Document Upload")

            # File upload component for PDF files
            file_upload = gr.File(
                label="Upload PDF Documents (supports 200+ pages)",
                file_count="multiple",
                file_types=[".pdf"],
                type="filepath"
            )

            # Button to trigger document processing
            process_btn = gr.Button("🔄 Process Documents", variant="primary", size="lg")

            gr.Markdown("### 📊 Processing Status")
            # Markdown component to display processing status
            status_display = gr.Markdown("Upload PDFs to begin processing...")

            gr.Markdown("---\n### ⚙️ Retrieval Settings")

            # Slider to control the number of chunks retrieved
            k_slider = gr.Slider(
                minimum=2,
                maximum=10,
                value=5,
                step=1,
                label="Chunks to Retrieve",
                info="More chunks = more context but slower"
            )

            gr.Markdown("---\n### 📈 System Statistics")
            # Button to view system statistics
            stats_btn = gr.Button("View Statistics", variant="secondary")

            # Button to reset the system
            reset_btn = gr.Button("🗑️ Reset System", variant="stop")

        # RIGHT: Chat Interface Column
        with gr.Column(scale=2):
            gr.Markdown("### 💬 Ask Questions About Your Documents")

            # Chatbot component for conversation history
            chatbot = gr.Chatbot(
                label="Conversation",
                height=550,
                show_copy_button=True
            )

            # Row for message input and send button
            with gr.Row():
                # Textbox for user input
                msg_input = gr.Textbox(
                    placeholder="Ask questions about mortgage terms, loan details, borrower information...",
                    label="Your Question",
                    scale=5,
                    show_label=False
                )
                # Button to send the message
                send_btn = gr.Button("📤 Send", variant="primary", scale=1)

            # Button to clear the chat history
            clear_chat_btn = gr.Button("Clear Chat")

            # Examples to pre-populate the input box
            gr.Examples(
                examples=[
                    ["What is the loan amount?"],
                    ["Who is the borrower?"],
                    ["What are the key terms of this mortgage?"],
                    ["What fees are listed in this document?"],
                    ["What is the interest rate?"],
                    ["Summarize the main points of this agreement"],
                ],
                inputs=msg_input
            )

    # Markdown for system capabilities
    gr.Markdown("""
    ---
    ### 🎯 System Capabilities

    **PDF Processing:**
    - Handles 200+ page documents reliably
    - Multi-tier extraction: PyMuPDF → pdfplumber → PyPDF2 → Tesseract OCR → EasyOCR
    - Intelligent fallback ensures text extraction even from poor-quality scans
    - Table detection and extraction
    - Real-time progress tracking with time estimation

    **Text Analysis:**
    - Semantic chunking preserves context across sentence boundaries
    - Page-aware citations for precise source tracking
    - Confidence scoring at page and chunk level
    - Metadata tracking: extraction method, processing time, quality metrics

    **Retrieval & Generation:**
    - State-of-the-art embeddings (sentence-transformers)
    - FAISS vector similarity search
    - Groq Llama 3 70B for high-quality responses
    - Automatic model fallback for reliability
    - Source attribution with confidence scores

    **Technical Stack:**
    - Embeddings: sentence-transformers/all-mpnet-base-v2
    - Vector Store: FAISS
    - LLM: Groq (Llama 3 70B / Mixtral 8x7B)
    - PDF: PyMuPDF, pdfplumber, PyPDF2
    - OCR: Tesseract, EasyOCR with CV preprocessing
    """)

    # Define event handlers for UI components
    process_btn.click(
        fn=process_pdf_ui, # Function to call
        inputs=file_upload, # Input component
        outputs=status_display # Output component
    )

    send_btn.click(
        fn=chat_ui, # Function to call
        inputs=[msg_input, chatbot, k_slider], # Input components
        outputs=chatbot # Output component
    ).then(
        lambda: "", # Clear the input box after sending
        outputs=msg_input
    )

    msg_input.submit( # Handle pressing Enter in the input box
        fn=chat_ui,
        inputs=[msg_input, chatbot, k_slider],
        outputs=chatbot
    ).then(
        lambda: "",
        outputs=msg_input
    )

    clear_chat_btn.click(
        lambda: [], # Clear the chatbot history
        outputs=chatbot
    )

    stats_btn.click(
        fn=get_stats_ui, # Function to get and display stats
        outputs=status_display
    )

    reset_btn.click(
        fn=clear_system, # Function to reset the system
        outputs=[chatbot, status_display] # Clear chat and status display
    )

# Launch the Gradio interface
demo.launch(share=True, debug=True)

/tmp/ipython-input-2116205693.py:55: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://627ef98bebbc1428f2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d7a5dcd653c2d9b91f.gradio.live
Killing tunnel 127.0.0.1:7860 <> https://627ef98bebbc1428f2.gradio.live
